# <font color = "#009d71">**Trabalho PI - inserção de dados com PyMongo**
### **Pontifícia Universidade Católica de Campinas**
### **Prof. Felipe Cavalaro**
### **Integrantes**
- Francisco da Silva Bueno Junior - 25008051
- Isabel Baungartner - 25001436
- Italo Fraga Botelho -
- Tiago Noda Von Zuben - 25018493

In [1]:
!pip install pymongo

!pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 27.3 MB/s eta 0:00:00


In [2]:
#importando bibliotecas
import pymongo
import pandas as pd
import numpy as np
import geopandas as gpd
import json

In [3]:
# conexão com MongoDB
client = pymongo.MongoClient(
"mongodb+srv://<USER>:<PASSWORD>@cluster0.a6z6ys0.mongodb.net/?appName=Cluster0")

# banco de dados
db = client["siest_db"]

### **1. Coleção de casos geolocalizados**

In [ ]:
# Lê o arquivo Parquet gerado na etapa de geolocalização
casos = pd.read_parquet("/content/casos_geolocalizados.parquet")

# Trazendo todas as colunas essenciais
casos = casos[
    [
        "NOME_DOENCA",
        "ID_UNIDADE",
        "LATITUDE",
        "LONGITUDE",
        "ID_MUNICIP",
        "DT_NOTIFIC",
        "DT_SIN_PRI",
        "NU_IDADE_N",
        "CS_SEXO",
        "HOSPITALIZ",
        "CLASSI_FIN",
        "EVOLUCAO",
        "CRITERIO",
        "NO_FANTASIA",
    ]
]

# Força a idade a ser um número puro no Python
casos["NU_IDADE_N"] = pd.to_numeric(casos["NU_IDADE_N"], errors="coerce")

# Tratamento das colunas de tempo
casos["DT_NOTIFIC"] = pd.to_datetime(casos["DT_NOTIFIC"], errors="coerce")
casos["DT_SIN_PRI"] = pd.to_datetime(casos["DT_SIN_PRI"], errors="coerce")

# CORREÇÃO PARA O PYMONGO: Substituir pandas.NaT e np.nan por None
casos = casos.replace({np.nan: None})

# Limpa a coleção antes de inserir os dados corrigidos
db.casos_geolocalizados.drop()

# Inserção
db.casos_geolocalizados.insert_many(casos.to_dict("records"))
print("Base inserida com sucesso e com todas as colunas clínicas!")

Base inserida com sucesso e com todas as colunas clínicas!


### **2. Coleção do clima de campinas entre 2014 e 2026**

In [ ]:
# 1. Carregamento do arquivo Parquet
clima = pd.read_parquet("/content/clima_campinas_2014_2026.parquet")

# 2. Seleção Atualizada de TODAS as variáveis de interesse
colunas_climaticas = [
    "DT_MEDICAO",
    "TEMP_MIN",
    "TEMP_MED",
    "TEMP_MAX",
    "PRECIP_TOTAL",
    "UMIDADE_MED"
]
clima = clima[colunas_climaticas]

# 3. Tratamento de Datas
clima["DT_MEDICAO"] = pd.to_datetime(clima["DT_MEDICAO"], errors="coerce")
clima = clima.dropna(subset=["DT_MEDICAO"]) # Remove qualquer linha com data corrompida

# 4. IMPUTAÇÃO DE SENSORES CEGOS (Tratamento Expandido)
# 1. TEMPERATURA E UMIDADE: Interpolação Linear (Curva Suave)
# Em vez de repetir cegamente, traçamos uma reta entre o dia anterior e o próximo dia válido
colunas_continuas = ["TEMP_MIN", "TEMP_MED", "TEMP_MAX", "UMIDADE_MED"]
clima[colunas_continuas] = clima[colunas_continuas].interpolate(method='linear', limit_direction='both')

# 2. PRECIPITAÇÃO: Conservadorismo Zero (Eventos Discretos)
# Evita criar tempestades fantasmas através de médias matemáticas.
clima["PRECIP_TOTAL"] = clima["PRECIP_TOTAL"].fillna(0)

# Preenche dias sem registo de precipitação com 0 (Significa que não choveu)
clima["PRECIP_TOTAL"] = clima["PRECIP_TOTAL"].fillna(0)

# 5. PROTEÇÃO DO MONGODB (Conversão de Tipos)
# O MongoDB recusa valores 'NaN'. Convertê-los para 'None' resolve o problema de tipagem
clima = clima.replace({np.nan: None})

# 6. INSERÇÃO IDEMPOTENTE
print("A limpar a coleção antiga para evitar duplicação de dados...")
db.clima.delete_many({}) # Limpa a coleção antes de inserir os novos

print(f"A inserir {len(clima)} registros climáticos com o conjunto de variáveis atualizado...")
db.clima.insert_many(clima.to_dict("records"))

print("✅ Inserção Climática concluída com sucesso no MongoDB!")

A limpar a coleção antiga para evitar duplicação de dados...
A inserir 4512 registros climáticos com o conjunto de variáveis atualizado...
✅ Inserção Climática concluída com sucesso no MongoDB!


### **3. Coleção do risco de inundação E vunerabilidade habitacional**

In [5]:
risco = gpd.read_parquet("/content/risco_inundacao_campinas.parquet")
vulnerabilidade = gpd.read_parquet("/content/vulnerabilidade_habitacional_campinas.parquet")

risco = risco[["geometry", "CLASSE"]]
vulnerabilidade = vulnerabilidade[["geometry", "NOME_AREA"]]

print("A processar a simplificação de nós duplicados...")
risco['geometry'] = risco['geometry'].simplify(0.00001, preserve_topology=True).buffer(0)
vulnerabilidade['geometry'] = vulnerabilidade['geometry'].simplify(0.00001, preserve_topology=True).buffer(0)

# Filtrar apenas as geometrias que são estritamente válidas após o curativo
risco = risco[~risco.is_empty & risco.is_valid]
vulnerabilidade = vulnerabilidade[~vulnerabilidade.is_empty & vulnerabilidade.is_valid]

def preparar_para_mongodb(gdf):
    geojson_str = gdf.to_json()
    geojson_dict = json.loads(geojson_str)
    return geojson_dict['features']

print("A limpar coleções espaciais antigas no MongoDB...")
db.risco_inundacao.delete_many({})
db.vulnerabilidade_habitacional.delete_many({})

print("A inserir Risco de Inundação com mapas curados...")
db.risco_inundacao.insert_many(preparar_para_mongodb(risco))

print("A inserir Vulnerabilidade Habitacional com mapas curados...")
db.vulnerabilidade_habitacional.insert_many(preparar_para_mongodb(vulnerabilidade))

A processar a simplificação de nós duplicados...
A limpar coleções espaciais antigas no MongoDB...
A inserir Risco de Inundação com mapas curados...
A inserir Vulnerabilidade Habitacional com mapas curados...


InsertManyResult([ObjectId('6a11d7bd6affb8d75990b3d7'), ObjectId('6a11d7bd6affb8d75990b3d8'), ObjectId('6a11d7bd6affb8d75990b3d9'), ObjectId('6a11d7bd6affb8d75990b3da'), ObjectId('6a11d7bd6affb8d75990b3db'), ObjectId('6a11d7bd6affb8d75990b3dc'), ObjectId('6a11d7bd6affb8d75990b3dd'), ObjectId('6a11d7bd6affb8d75990b3de'), ObjectId('6a11d7bd6affb8d75990b3df'), ObjectId('6a11d7bd6affb8d75990b3e0'), ObjectId('6a11d7bd6affb8d75990b3e1'), ObjectId('6a11d7bd6affb8d75990b3e2'), ObjectId('6a11d7bd6affb8d75990b3e3'), ObjectId('6a11d7bd6affb8d75990b3e4'), ObjectId('6a11d7bd6affb8d75990b3e5'), ObjectId('6a11d7bd6affb8d75990b3e6'), ObjectId('6a11d7bd6affb8d75990b3e7'), ObjectId('6a11d7bd6affb8d75990b3e8'), ObjectId('6a11d7bd6affb8d75990b3e9'), ObjectId('6a11d7bd6affb8d75990b3ea'), ObjectId('6a11d7bd6affb8d75990b3eb'), ObjectId('6a11d7bd6affb8d75990b3ec'), ObjectId('6a11d7bd6affb8d75990b3ed'), ObjectId('6a11d7bd6affb8d75990b3ee'), ObjectId('6a11d7bd6affb8d75990b3ef'), ObjectId('6a11d7bd6affb8d75990b3